In [2]:
import pickle
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, RepeatedKFold, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from valuation_utils import build_features, Winsorizer, SectorMedianRegressor, NUM_FEATURES, CAT_FEATURES, FEATURES


def make_rf_pipeline():
    preprocessor = ColumnTransformer([
        ('num', Pipeline([
            ('win', Winsorizer()),
            ('imp', SimpleImputer(strategy='median', add_indicator=True))
        ]), NUM_FEATURES),
        ('cat', OneHotEncoder(handle_unknown='ignore'), CAT_FEATURES),
    ])
    rf = RandomForestRegressor(n_estimators=300, max_depth=5, min_samples_leaf=5, random_state=42, n_jobs=-1)
    return TransformedTargetRegressor(
        Pipeline([('pre', preprocessor), ('rf', rf)]),
        func=np.log, inverse_func=np.exp
    )


def train_and_evaluate(input_csv="nifty500_clean_v2.csv", model_output="nifty_valuation_model.pkl"):
    df = build_features(pd.read_csv(input_csv))
    X, y = df[FEATURES], df['Trailing_PE']
    print(f"Dataset sample size: {len(df)} companies")

    cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)
    scoring = {'mae': 'neg_mean_absolute_error', 'medae': 'neg_median_absolute_error', 'r2': 'r2'}
    results = {}

    for name, est in [('Sector Median', SectorMedianRegressor()), ('Random Forest', make_rf_pipeline())]:
        r = cross_validate(est, X, y, cv=cv, scoring=scoring)
        results[name] = r
        print(f"{name:16s} MAE {-r['test_mae'].mean():6.2f} | "
              f"MedAE {-r['test_medae'].mean():6.2f} | "
              f"R2 {r['test_r2'].mean():6.3f} (±{r['test_r2'].std():.3f})")

    base, rf_res = results['Sector Median'], results['Random Forest']
    win_rate = (-rf_res['test_medae'] < -base['test_medae']).mean()
    print(f"\nRandom Forest beats Sector Median (MedAE) in {win_rate:.0%} of folds.")

    oof = cross_val_predict(make_rf_pipeline(), X, y, cv=KFold(5, shuffle=True, random_state=42))
    log_residuals = np.log(y.values / oof)
    q_lo, q_hi = np.quantile(log_residuals, [0.10, 0.90])
    print(f"80% P/E interval multiplier: x{np.exp(q_lo):.2f} .. x{np.exp(q_hi):.2f} of Fair P/E")

    final_model = make_rf_pipeline().fit(X, y)
    with open(model_output, "wb") as f:
        pickle.dump({'model': final_model, 'q_lo': q_lo, 'q_hi': q_hi, 'features': FEATURES}, f)
    print(f"Model successfully saved to '{model_output}'.")


# Esegui l'addestramento nel notebook
train_and_evaluate()

Dataset sample size: 474 companies
Sector Median    MAE  25.77 | MedAE  13.57 | R2 -0.027 (±0.054)
Random Forest    MAE  22.47 | MedAE  12.28 | R2  0.207 (±0.075)

Random Forest beats Sector Median (MedAE) in 80% of folds.
80% P/E interval multiplier: x0.48 .. x2.19 of Fair P/E
Model successfully saved to 'nifty_valuation_model.pkl'.
